
# DISL Invariant Clustering (Require vs Assert)

This notebook clusters **require** and **assert** statements from the DISL *decomposed* dataset (Solidity only), using the Parquet outputs created by `extract_disl_guards.py`.  
It mirrors the structure and function names from your previous clustering notebook so you can drop it into your workflow easily.

**What you get:**
- Clean loaders for `disl_requires.parquet` and `disl_asserts.parquet`
- Dedup options by `normalized_pred`
- Text corpora construction (with/without error message for `require`)
- TF‑IDF baseline + optional transformer embeddings
- PCA/UMAP normalization + reduction pipeline
- KMeans, DBSCAN, HDBSCAN sweeps with **Silhouette** and **S_Dbw**
- CSVs with metrics and cluster labels, plus handy preview utilities


In [ ]:

# Optional: install dependencies if needed 
!pip install umap-learn hdbscan s_dbw transformers torch scikit-learn --upgrade

In [7]:

import os, math, re, json, gc
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline

import hdbscan
from umap import UMAP

# S_Dbw score
try:
    from s_dbw import S_Dbw
except Exception as e:
    print("s_dbw import failed. If needed, run the install cell above.")

# Thresholds (similar to previous notebook)
POINTS_THRESHOLD = 0.50
MIN_CLUSTER = 8
MAX_CLUSTER = 100
RANDOM_SEED = 42

pd.set_option("display.max_colwidth", 180)


/Users/mojtabae/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

# === Paths (edit me) ===
DATA_DIR = "out"   # directory where extract_disl_guards.py wrote the parquet files
REQUIRES_PARQUET = os.path.join(DATA_DIR, "disl_requires.parquet")
ASSERTS_PARQUET  = os.path.join(DATA_DIR, "disl_asserts.parquet")

OUT_DIR = "clustering_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("Looking for:", REQUIRES_PARQUET, ASSERTS_PARQUET)


Looking for: out/disl_requires.parquet out/disl_asserts.parquet


In [9]:

def load_disl_frame(path):
    df = pd.read_parquet(path)
    # Keep essential columns; pass others through so we can trace back
    expected_cols = [
        "contract_address","file_path","compiler_version","license_type","contract_name",
        "statement_kind","statement_index","full_statement","predicate","message",
        "normalized_pred","line_start","col_start","file_sha1"
    ]
    keep = [c for c in expected_cols if c in df.columns]
    return df[keep].copy()

def dedupe_by_normalized_pred(df):
    # Deduplicate on normalized predicate, but keep a representative row
    # Also collect occurrences so we can trace back later
    agg = (df.groupby("normalized_pred", dropna=False)
             .agg(
                 n_occ=("normalized_pred","size"),
                 any_kind=("statement_kind","first"),
                 sample_pred=("predicate","first"),
                 sample_msg=("message","first"),
                 sample_full=("full_statement","first"),
                 sample_file_sha1=("file_sha1","first"),
             )
             .reset_index())
    return agg

# Match your prior df_to_string() behavior
def df_to_string_from_agg(agg_df, with_message=True):
    lines = []
    for _, row in agg_df.iterrows():
        predicate = str(row["sample_pred"]) if pd.notnull(row["sample_pred"]) else ""
        message = row.get("sample_msg", None)
        if with_message and pd.notnull(message):
            line = f"{predicate}, '{message}'"
        else:
            line = f"{predicate},"
        lines.append(line)
    return lines

def df_to_string_asserts(agg_df):
    # Asserts don't have messages, just the predicate
    lines = []
    for _, row in agg_df.iterrows():
        predicate = str(row["sample_pred"]) if pd.notnull(row["sample_pred"]) else ""
        lines.append(f"{predicate},")
    return lines

def string_length_stats(strings):
    if not strings: 
        return {"average": 0, "min": 0, "max": 0}
    lens = [len(s) for s in strings]
    return {"average": sum(lens)/len(lens), "min": min(lens), "max": max(lens)}


In [10]:

req_df = load_disl_frame(REQUIRES_PARQUET)
asr_df = load_disl_frame(ASSERTS_PARQUET)

print(f"Requires rows: {len(req_df):,} | Asserts rows: {len(asr_df):,}")
display(req_df.head(2))
display(asr_df.head(2))

# Deduplicate at the predicate level (you can switch to full_statement if you prefer)
req_agg = dedupe_by_normalized_pred(req_df)
asr_agg = dedupe_by_normalized_pred(asr_df)

print(f"Unique require predicates: {len(req_agg):,}")
print(f"Unique assert predicates:  {len(asr_agg):,}")


Requires rows: 5,276,315 | Asserts rows: 105,452


,contract_address,file_path,compiler_version,license_type,contract_name,statement_kind,statement_index,full_statement,predicate,message,normalized_pred,line_start,col_start,file_sha1
0,0x1994c717ec123fe2830399fe4f0f3054c8e69ebc,OwnableDelegateProxy.sol,v0.4.23+commit.124ca40d,MIT,OwnableDelegateProxy,require,0,require(_impl != address(0));,_impl != address(0),None,_impl != address(0),63,5,0e616938c4b4d45ec9ed148c46b2c398673e4c10
1,0x1994c717ec123fe2830399fe4f0f3054c8e69ebc,OwnableDelegateProxy.sol,v0.4.23+commit.124ca40d,MIT,OwnableDelegateProxy,require,1,require(_implementation != implementation);,_implementation != implementation,None,_implementation != implementation,98,5,0e616938c4b4d45ec9ed148c46b2c398673e4c10


,contract_address,file_path,compiler_version,license_type,contract_name,statement_kind,statement_index,full_statement,predicate,message,normalized_pred,line_start,col_start,file_sha1
0,0x6401e2ff943722e8583d0f7b99987e7b39f9cc75,contracts/methodNFT/MethodVaultV2.sol,v0.7.6+commit.7338295f,,MethodVaultV2,assert,0,assert(_lockSet.add(lockID));,_lockSet.add(lockID),None,_lockSet.add(lockID),395,13,a491382026e0847d9bb4290501118185f3779a6c
1,0x6401e2ff943722e8583d0f7b99987e7b39f9cc75,contracts/methodNFT/MethodVaultV2.sol,v0.7.6+commit.7338295f,,MethodVaultV2,assert,5,assert(_lockSet.add(lockID));,_lockSet.add(lockID),None,_lockSet.add(lockID),456,13,a491382026e0847d9bb4290501118185f3779a6c


Unique require predicates: 399,683
Unique assert predicates:  5,863


In [11]:

with_msg_corpus    = df_to_string_from_agg(req_agg, with_message=True)
without_msg_corpus = df_to_string_from_agg(req_agg, with_message=False)
asserts_corpus     = df_to_string_asserts(asr_agg)

print("Require (with message) stats:", string_length_stats(with_msg_corpus))
print("Require (no message) stats:", string_length_stats(without_msg_corpus))
print("Assert stats:", string_length_stats(asserts_corpus))


Require (with message) stats: {'average': 67.22630184421153, 'min': 2, 'max': 33516}
Require (no message) stats: {'average': 40.94874688190391, 'min': 2, 'max': 33487}
Assert stats: {'average': 41.61453180965376, 'min': 3, 'max': 718}


In [12]:

def make_tfidf_char_ngrams(corpus, ngram_range=(3,6), min_df=2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram_range, min_df=min_df)
    X = vec.fit_transform(corpus)
    return X, vec

def make_counts_char_ngrams(corpus, ngram_range=(3,6), min_df=2):
    vec = CountVectorizer(analyzer="char_wb", ngram_range=ngram_range, min_df=min_df)
    X = vec.fit_transform(corpus)
    return X, vec


In [13]:

# OPTIONAL: Hugging Face encoder embeddings (CodeBERT/SmartBERT/ReBERT names go here)
# If you don't need encoder models, skip this cell and use TF‑IDF counts/TF‑IDF vectors above.

def embedding_model(texts, name):
    from transformers import RobertaTokenizer, RobertaModel
    import torch

    tokenizer = RobertaTokenizer.from_pretrained(name)
    model = RobertaModel.from_pretrained(name)
    model.eval()

    embs = []
    for t in texts:
        with torch.no_grad():
            toks = tokenizer(
                t if isinstance(t, str) else str(t),
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding="max_length",
            )
            out = model(**toks)
            # CLS-pooled (768-d) – simple baseline; replace with mean-pooling if you prefer
            e = out.last_hidden_state[:,0,:].squeeze(0).cpu().numpy()
            embs.append(e)
    return np.vstack(embs)

# Examples (commented): pick the ones you want and run
# codeBERT_emb   = embedding_model(with_msg_corpus,   "microsoft/codebert-base")
# smartBERT_emb  = embedding_model(with_msg_corpus,   "ASSERT-KTH/SmartBERT-v2")
# reBERT_emb     = embedding_model(with_msg_corpus,   "your-hf-org/ReBERT")


In [14]:

def normalize_and_reduce(data, use_umap=False, umap_components=20, pca_components=10):
    # Accepts dense (np.ndarray) or sparse matrix
    if use_umap:
        reducer = UMAP(n_components=umap_components, metric="cosine", random_state=RANDOM_SEED)
        if hasattr(data, "toarray"):
            data = data.toarray()
        data = Normalizer(norm="l2").fit_transform(data)
        return reducer.fit_transform(data)
    else:
        pipeline = make_pipeline(Normalizer(norm="l2"), PCA(n_components=pca_components, random_state=RANDOM_SEED))
        if hasattr(data, "toarray"):
            data = data.toarray()
        return pipeline.fit_transform(data)


In [15]:

def cluster_evaluate(labels, data, label, metric="cosine"):
    labels = np.array(labels)
    valid_mask = labels != -1
    valid_data = data[valid_mask]
    valid_labels = labels[valid_mask]

    pct_clustered = len(valid_labels) / len(labels) if len(labels) else 0.0
    num_clusters = len(set(valid_labels)) if len(set(valid_labels)) > 1 else 0
    sil_score = -1
    sdbw_score = 1e9  # large means bad

    if pct_clustered > POINTS_THRESHOLD and MIN_CLUSTER <= num_clusters <= MAX_CLUSTER:
        try:
            sil_score = silhouette_score(valid_data, valid_labels, metric=metric)
        except Exception as e:
            sil_score = -1
        try:
            sdbw_score = S_Dbw(valid_data, valid_labels, metric="cosine", alg_noise="bind", centr="mean")
        except Exception as e:
            sdbw_score = 1e9

    row = dict(
        label=label,
        silhouette=sil_score,
        s_dbw=sdbw_score,
        num_clusters=num_clusters,
        pct_clustered=pct_clustered,
    )
    return row

def cluster_search_kmeans(data, label, k_values):
    results = []
    for k in k_values:
        km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto")
        y = km.fit_predict(data)
        row = cluster_evaluate(y, data, f"{label}|kmeans|k={k}", metric="cosine")
        row["algorithm"] = "kmeans"
        row["params"] = {"k": k}
        results.append(row)
    return pd.DataFrame(results).sort_values(["s_dbw","silhouette"], ascending=[True,False])

def cluster_search_dbscan(data, label, eps_values, min_samples_values):
    results = []
    for eps in eps_values:
        for ms in min_samples_values:
            db = DBSCAN(eps=eps, min_samples=ms, metric="cosine", n_jobs=-1)
            y = db.fit_predict(data)
            row = cluster_evaluate(y, data, f"{label}|dbscan|eps={eps}|ms={ms}", metric="cosine")
            row["algorithm"] = "dbscan"
            row["params"] = {"eps": eps, "min_samples": ms}
            results.append(row)
    return pd.DataFrame(results).sort_values(["s_dbw","silhouette"], ascending=[True,False])

def cluster_search_hdbscan(data, label, min_cluster_sizes):
    results = []
    for mcs in min_cluster_sizes:
        hb = hdbscan.HDBSCAN(min_cluster_size=mcs, metric="euclidean")  # UMAP/PCA has already normalized
        y = hb.fit_predict(data)
        row = cluster_evaluate(y, data, f"{label}|hdbscan|mcs={mcs}", metric="cosine")
        row["algorithm"] = "hdbscan"
        row["params"] = {"min_cluster_size": mcs}
        results.append(row)
    return pd.DataFrame(results).sort_values(["s_dbw","silhouette"], ascending=[True,False])


In [16]:

def run_all_clustering(X, label_prefix, grids=None, use_umap=False):
    # X: sparse or dense features
    # label_prefix: string to prefix result rows and saved files
    # grids: dict to override default ranges
    # use_umap: use UMAP instead of PCA in normalize_and_reduce
    if grids is None:
        grids = {
            "kmeans_k": list(range(8, 101, 3)),
            "dbscan_eps": [0.1, 0.2, 0.3, 0.36, 0.4, 0.6, 0.8],
            "dbscan_min_samples": [5, 8, 10, 12, 14, 18],
            "hdbscan_mcs": list(range(8, 31, 2)),
        }

    Xr = normalize_and_reduce(X, use_umap=use_umap, umap_components=20, pca_components=10)

    df_km  = cluster_search_kmeans(Xr, label_prefix, grids["kmeans_k"])
    df_db  = cluster_search_dbscan(Xr, label_prefix, grids["dbscan_eps"], grids["dbscan_min_samples"])
    df_hdb = cluster_search_hdbscan(Xr, label_prefix, grids["hdbscan_mcs"])

    results = (pd.concat([df_km, df_db, df_hdb], ignore_index=True)
                 .sort_values(["s_dbw","silhouette"], ascending=[True,False]))

    out_csv = os.path.join(OUT_DIR, f"{label_prefix}_grid_results.csv")
    results.to_csv(out_csv, index=False)
    print(f"Saved grid results to: {out_csv}")
    display(results.head(15))
    return Xr, results


In [17]:

# === REQUIRE (with error message) ===
X_req_tfidf_with, vec_req_with = make_tfidf_char_ngrams(with_msg_corpus, ngram_range=(3,6), min_df=2)
Xr_req_with, res_req_with = run_all_clustering(X_req_tfidf_with, "require_withmsg_tfidf", use_umap=False)

# === REQUIRE (without error message) ===
X_req_tfidf_without, vec_req_without = make_tfidf_char_ngrams(without_msg_corpus, ngram_range=(3,6), min_df=2)
Xr_req_without, res_req_without = run_all_clustering(X_req_tfidf_without, "require_nomsg_tfidf", use_umap=False)

# === ASSERT (predicate only) ===
X_asr_tfidf, vec_asr = make_tfidf_char_ngrams(asserts_corpus, ngram_range=(3,6), min_df=2)
Xr_asr, res_asr = run_all_clustering(X_asr_tfidf, "assert_tfidf", use_umap=False)


: 

In [ ]:

def fit_and_label_best(Xr, results_df, label_prefix):
    # Pick the top row with finite S_Dbw and best silhouette
    cand = results_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["s_dbw","silhouette"]).head(1)
    if cand.empty:
        cand = results_df.head(1)
    row = cand.iloc[0]
    algo = row["algorithm"]
    params = row["params"] if isinstance(row["params"], dict) else {}

    if algo == "kmeans":
        km = KMeans(n_clusters=int(params["k"]), random_state=RANDOM_SEED, n_init="auto").fit(Xr)
        labels = km.labels_
    elif algo == "dbscan":
        db = DBSCAN(eps=float(params["eps"]), min_samples=int(params["min_samples"]), metric="cosine", n_jobs=-1).fit(Xr)
        labels = db.labels_
    elif algo == "hdbscan":
        hb = hdbscan.HDBSCAN(min_cluster_size=int(params["min_cluster_size"]), metric="euclidean").fit(Xr)
        labels = hb.labels_
    else:
        raise ValueError(f"Unknown algorithm: {algo}")

    print(f"Best: {algo} {params}")
    out_csv = os.path.join(OUT_DIR, f"{label_prefix}_labels.csv")
    pd.DataFrame({"label": labels}).to_csv(out_csv, index=False)
    print(f"Saved labels to: {out_csv}")
    return np.array(labels), algo, params


In [ ]:

# REQUIRE with message
labels_req_with, algo1, p1 = fit_and_label_best(Xr_req_with, res_req_with, "require_withmsg_tfidf")
req_with_out = req_agg.copy()
req_with_out["label"] = labels_req_with
req_with_out.to_csv(os.path.join(OUT_DIR, "require_withmsg_clusters.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "require_withmsg_clusters.csv"))

# REQUIRE without message
labels_req_without, algo2, p2 = fit_and_label_best(Xr_req_without, res_req_without, "require_nomsg_tfidf")
req_without_out = req_agg.copy()
req_without_out["label"] = labels_req_without
req_without_out.to_csv(os.path.join(OUT_DIR, "require_nomsg_clusters.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "require_nomsg_clusters.csv"))

# ASSERT
labels_asr, algo3, p3 = fit_and_label_best(Xr_asr, res_asr, "assert_tfidf")
asr_out = asr_agg.copy()
asr_out["label"] = labels_asr
asr_out.to_csv(os.path.join(OUT_DIR, "assert_clusters.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "assert_clusters.csv"))


In [ ]:

def preview_clusters(df_with_labels, k=8):
    sample = (df_with_labels[df_with_labels["label"]!=-1]
                .groupby("label")
                .head(1)[["label","normalized_pred","sample_pred","sample_msg","n_occ"]])
    display(sample.head(k))

print("\n=== REQUIRE (with message) sample clusters ===")
preview_clusters(req_with_out)

print("\n=== REQUIRE (no message) sample clusters ===")
preview_clusters(req_without_out)

print("\n=== ASSERT sample clusters ===")
preview_clusters(asr_out)
